# Fake News Detector — Group 14C  
### AI4ALL Ignite — Fall 2024

**Authors:** Wynne Conger, Nina Elmoyan, Ramneek Kaur, Rhode Sanchez  
**Goal:** Build linguistically informed features to distinguish fake vs. real news.


## 1. Imports & Setup

In [ ]:
# Imports & Setup
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")
nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords

from better_profanity import profanity
profanity.load_censor_words()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, RocCurveDisplay, roc_auc_score
)
from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

from sklearn.utils import shuffle

# Initialize tools
sia = SentimentIntensityAnalyzer()
EN_STOPWORDS = set(stopwords.words("english"))

RANDOM_STATE = 42


## 2. Load Dataset
The dataset contains `Fake.csv` and `True.csv`. We label and combine them later.


In [ ]:
df_fake = pd.read_csv("Dataset/Fake.csv")
df_true = pd.read_csv("Dataset/True.csv")

df_fake["label"] = 0
df_true["label"] = 1

## 3. Helper Functions
Reusable functions for feature extraction.

In [ ]:
def apply_feature(df_fake, df_true, feature_name, func, col="text"):
    df_fake[feature_name] = df_fake[col].apply(func)
    df_true[feature_name] = df_true[col].apply(func)


def lexical_diversity_optimized(text):
    tokens = nltk.word_tokenize(str(text).lower())
    words = [w for w in tokens if w.isalpha()]
    if not words:
        return 0
    return len(set(words)) / len(words)


bad_words_set = {w.lower() for w in profanity.CENSOR_WORDSET}

def fast_profanity_count(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for w in text.lower().split() if w in bad_words_set)


def count_pronouns(text):
    first = {'i','we','me','us','my','our','mine','ours'}
    others = {'you','he','she','they','him','her','them','your','his','its','their'}
    words = nltk.word_tokenize(str(text).lower())
    fp = sum(w in first for w in words)
    op = sum(w in others for w in words)
    return pd.Series([fp, op])


## 4. Feature Dictionary
Each feature is defined as a function.  
This makes the notebook scalable and clean.

FEATURES = {
    "word_count": lambda x: len(nltk.word_tokenize(x)),
    "question_marks": lambda x: x.count("?"),
    "negativity": lambda x: sia.polarity_scores(x)["neg"],
    "text_sentiment": lambda x: sia.polarity_scores(x)["compound"],
    "lexical_diversity": lexical_diversity_optimized,
    "profanity_count": fast_profanity_count,
}

## 5. Apply Linguistic Features
This section computes all features for both fake and real datasets.

In [ ]:
for name, func in FEATURES.items():
    apply_feature(df_fake, df_true, name, func)

# Pronouns added separately
df_fake[["first_person", "others"]] = df_fake["text"].apply(count_pronouns)
df_true[["first_person", "others"]] = df_true["text"].apply(count_pronouns)

## 6. Combine Datasets for Analysis

In [ ]:
df_fake["is_fake"] = 1
df_true["is_fake"] = 0

df_combined = pd.concat([df_fake, df_true], ignore_index=True)

## 7. Statistical Tests (t-tests)
Testing whether each linguistic feature significantly differs between fake and real news.


In [ ]:
feature_list = list(FEATURES.keys()) + ["first_person", "others"]

ttest_results = {}

for f in feature_list:
    fake_vals = df_fake[f].dropna()
    true_vals = df_true[f].dropna()
    t, p = stats.ttest_ind(fake_vals, true_vals, equal_var=False)
    ttest_results[f] = (t, p)

ttest_results


## 8. Correlation Analysis
Understanding which linguistic features correlate most with the label.

In [ ]:
corr = df_combined.corr()
corr["is_fake"].sort_values(ascending=False)

## 9. Visualizations
Add plots here (correlation heatmap, boxplots, distributions).


## 10. Results, Observations, and Interpretation
Summaries of findings go here.

## 11. Appendix / Experimental Code
Scratch work, alternative models, and tests go here.